# Oscar Winner Prediction using Bayesian Networks

**Expert-level Implementation**

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.bayesian_network import OscarBayesianNetwork

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

---

## 1. Create Sample Dataset

In [ ]:
n_films = 150

data = pd.DataFrame({
    'Genre': np.random.choice(['Drama', 'Biography', 'Action', 'Comedy', 'Sci-Fi'], n_films),
    'IMDB_Rating': np.random.normal(7.5, 0.8, n_films).clip(5.5, 9.5).round(1),
    'Metacritic': np.random.randint(50, 100, n_films),
    'Box_Office': np.random.lognormal(18, 1.2, n_films).astype(int),
    'Director_Prestige': np.random.choice(['High', 'Medium', 'Low'], n_films, p=[0.3, 0.5, 0.2]),
    'Other_Awards_Won': np.random.poisson(2, n_films),
    'Oscar_Nominations': np.random.poisson(4, n_films),
})

oscar_wins = []
for _, row in data.iterrows():
    base = 0
    if row['Other_Awards_Won'] >= 5: base += 3
    if row['Oscar_Nominations'] >= 8: base += 4
    if row['IMDB_Rating'] > 8.0: base += 2
    if row['Director_Prestige'] == 'High': base += 1
    wins = max(0, int(np.random.normal(base, 2)))
    oscar_wins.append(min(wins, 11))

data['Oscar_Wins'] = oscar_wins

print("Sample data created successfully!")
print(data.head())

---

## 2. Build and Train the Bayesian Network

In [ ]:
bn = OscarBayesianNetwork()
bn.fit(data, use_expert_structure=True)

print("Bayesian Network Edges:")
print(bn.model.edges())

---

## 3. Predict for Oppenheimer (2024 Oscars)

In [ ]:
oppenheimer_evidence = {
    'Genre': 'Biography',
    'IMDB_Rating': 8.3,
    'Metacritic': 90,
    'Other_Awards_Won': 8,
    'Oscar_Nominations': 13,
    'Director_Prestige': 'High'
}

prediction = bn.predict_oscar_wins(oppenheimer_evidence)

print("=== Oppenheimer Prediction ===")
print(f"Predicted Oscar Wins: {prediction['predicted_wins']}")
print(f"Probability: {prediction['probability']:.2%}")
print(f"Actual Wins in 2024: 7")